# ML05：cross-attention compressor 与 frozen sidecar

训练一个合成上下文瓶颈，再冻结它预测 frame/mask/wrench；这是 FP2 思想的教学实例，不是真实 FP2 复现。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(SEED)
torch.set_num_threads(2)
print('torch:', torch.__version__, '| teaching device: CPU')


## 2. 先验证 causal mask 的信息依赖
这里 `nn.MultiheadAttention` 的 bool mask 中 True 表示禁止读取。扰动未来，过去输出应保持不变。


In [ ]:
torch.manual_seed(SEED)
attention = nn.MultiheadAttention(16, 4, dropout=0., batch_first=True).eval()
sequence = torch.randn(2, 6, 16)
causal_mask = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1)
changed = sequence.clone(); changed[:, 4:] += 100
with torch.no_grad():
    original = attention(sequence, sequence, sequence, attn_mask=causal_mask, need_weights=False)[0]
    perturbed = attention(changed, changed, changed, attn_mask=causal_mask, need_weights=False)[0]
causal_delta = float((original[:, :4]-perturbed[:, :4]).abs().max())
assert causal_delta < 1e-5
print('past-output delta after changing future:', causal_delta)


## 3. 合成 C 与交互 histories，按样本分 train/val/test
C 只含 task latent 的特征，不含 W/P。frame/mask/wrench 标签由已知合成规则生成；不使用论文数据或真实 RFM。


In [ ]:
generator = torch.Generator().manual_seed(SEED+20)
count, tokens, input_width, width, horizon = 768, 16, 32, 16, 10
task_latent = torch.randn(count, 6, generator=generator)
base = .2*torch.randn(tokens, input_width, generator=generator)
modes = .2*torch.randn(6, tokens, input_width, generator=generator)
C = base.unsqueeze(0) + torch.einsum('br,rnd->bnd', task_latent, modes)
contact = torch.randint(0, 2, (count,), generator=generator).float()
P = .05*torch.randn(count, horizon, 9, generator=generator)
P[:, :, 3:] += torch.tensor([1., 0., 0., 0., 1., 0.])
W = .02*torch.randn(count, horizon, 6, generator=generator)
W[:, :, 2] += contact[:, None]*(1.+4.*torch.rand(count, 1, generator=generator))
selection_target = torch.zeros(count, 6); selection_target[:, 2] = contact
wrench_target = torch.zeros(count, 6)
wrench_target[:, 2] = contact*(4.+2.*torch.sigmoid(task_latent[:, 0]))
angle = .5*torch.tanh(task_latent[:, 1])
c, s = torch.cos(angle), torch.sin(angle)
first_column = torch.stack([c, torch.zeros_like(c), -s], dim=-1)
second_column = torch.tensor([0., 1., 0.]).expand(count, -1)
frame_target = torch.cat([torch.zeros(count, 3), first_column, second_column], dim=-1)
order = torch.randperm(count, generator=generator)
train_ids, val_ids, test_ids = order[:512], order[512:640], order[640:]
print('C', C.shape, '| P', P.shape, '| W', W.shape)
print('targets:', frame_target.shape, selection_target.shape, wrench_target.shape)


## 4. Learned query → cross-attention → 单 token
Encoder 的 forward 只接收 C。decoder 的 N 个 queries 只读取 z；没有 C→decoder 信息旁路。


In [ ]:
class CrossBlock(nn.Module):
    def __init__(self, d=16, heads=4):
        super().__init__()
        self.qnorm = nn.LayerNorm(d); self.knorm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=0., batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 2*d), nn.GELU(), nn.Linear(2*d, d))
    def forward(self, query, context):
        kv = self.knorm(context)
        query = query + self.attn(self.qnorm(query), kv, kv, need_weights=False)[0]
        return query + self.mlp(self.norm(query))
class Compressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(input_width, width)
        self.query = nn.Parameter(.02*torch.randn(1, 1, width))
        self.blocks = nn.ModuleList([CrossBlock(width), CrossBlock(width)])
    def forward(self, context):
        x = self.projection(context)
        query = self.query.expand(len(context), -1, -1)
        for block in self.blocks:
            query = block(query, x)
        return query
class ReconstructionDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.queries = nn.Parameter(.1*torch.randn(1, tokens, width))
        self.block = CrossBlock(width)
        self.output = nn.Linear(width, input_width)
    def forward(self, z):
        queries = self.queries.expand(len(z), -1, -1)
        return self.output(self.block(queries, z))
compressor, decoder = Compressor(), ReconstructionDecoder()
with torch.no_grad():
    print('projected:', compressor.projection(C[:2]).shape,
          '| z:', compressor(C[:2]).shape, '| reconstruction:', decoder(compressor(C[:2])).shape)


## 5. 仅用 train C 训练重建
Loss=MSE+0.1×token-wise cosine distance。没有 force loss，未训练真实 RFM。


In [ ]:
COMPRESSOR_STEPS = 300
optimizer = torch.optim.AdamW(list(compressor.parameters())+list(decoder.parameters()), lr=.003)
reconstruction_trace = []
for step in range(COMPRESSOR_STEPS):
    batch = train_ids[torch.randint(len(train_ids), (64,))]
    target = C[batch]; reconstructed = decoder(compressor(target))
    objective = F.mse_loss(reconstructed, target) + .1*(1-F.cosine_similarity(reconstructed, target, dim=-1)).mean()
    optimizer.zero_grad(set_to_none=True); objective.backward(); optimizer.step()
    if step % 10 == 0:
        with torch.no_grad():
            val_mse = float(F.mse_loss(decoder(compressor(C[val_ids])), C[val_ids]))
        reconstruction_trace.append([step, float(objective.detach()), val_mse])
trace = np.asarray(reconstruction_trace)
plt.plot(trace[:, 0], trace[:, 2]); plt.xlabel('step'); plt.ylabel('validation reconstruction MSE'); plt.show()


## 6. 冻结 compressor，训练三头 force sidecar
这里选择 GRU+concat+MLP 作为教学配置。实际 FP2 主文没有完整公开该层配置。


In [ ]:
compressor.eval().requires_grad_(False)
for parameter in compressor.parameters():
    parameter.grad = None
with torch.no_grad():
    cached_z = compressor(C)
class ForceSidecar(nn.Module):
    def __init__(self):
        super().__init__()
        self.history = nn.GRU(15, 32, batch_first=True)
        self.fusion = nn.Sequential(nn.Linear(width+32, 64), nn.GELU())
        self.frame = nn.Linear(64, 9); self.mask = nn.Linear(64, 6); self.wrench = nn.Linear(64, 6)
    def forward(self, z, p_history, w_history):
        _, hidden = self.history(torch.cat([p_history, w_history], dim=-1))
        fused = self.fusion(torch.cat([z.squeeze(1), hidden[-1]], dim=-1))
        return self.frame(fused), self.mask(fused), self.wrench(fused)
sidecar = ForceSidecar(); optimizer = torch.optim.AdamW(sidecar.parameters(), lr=.003)
FORCE_STEPS = 300; force_trace = []
for step in range(FORCE_STEPS):
    batch = train_ids[torch.randint(len(train_ids), (64,))]
    frame, mask_logits, desired = sidecar(cached_z[batch], P[batch], W[batch])
    objective = (F.mse_loss(frame, frame_target[batch]) +
                 F.mse_loss(desired, wrench_target[batch]) +
                 .1*F.binary_cross_entropy_with_logits(mask_logits, selection_target[batch]))
    optimizer.zero_grad(set_to_none=True); objective.backward(); optimizer.step()
    if step % 10 == 0:
        force_trace.append([step, float(objective.detach())])
assert all(p.grad is None for p in compressor.parameters())
plt.plot(*np.asarray(force_trace).T); plt.xlabel('step'); plt.ylabel('training sidecar loss'); plt.show()


## 7. 固定模型后的 held-out 评价与 frame 解码
报告的是 synthetic labels 上的预测误差，不是力控成功率。6D rotation code 解码为 3×3 正交矩阵；9 个编码数不是 9 个物理自由度。


In [ ]:
sidecar.eval()
with torch.no_grad():
    frame, mask_logits, desired = sidecar(cached_z[test_ids], P[test_ids], W[test_ids])
    mask = (mask_logits.sigmoid() >= .5).float()
    first = F.normalize(frame[:, 3:6], dim=-1)
    second_raw = frame[:, 6:9] - (frame[:, 6:9]*first).sum(-1, keepdim=True)*first
    second = F.normalize(second_raw, dim=-1)
    third = torch.linalg.cross(first, second, dim=-1)
    rotation = torch.stack([first, second, third], dim=-1)
    results = {'seed': SEED, 'causal_perturbation_delta': causal_delta,
        'test_reconstruction_mse': float(F.mse_loss(decoder(cached_z[test_ids]), C[test_ids])),
        'test_frame_code_mse': float(F.mse_loss(frame, frame_target[test_ids])),
        'test_contact_axis_mask_accuracy': float((mask[:, 2]==selection_target[test_ids, 2]).float().mean()),
        'test_desired_wrench_mse': float(F.mse_loss(desired, wrench_target[test_ids])),
        'rotation_shape': list(rotation.shape),
        'compressor_frozen': all(not p.requires_grad for p in compressor.parameters()),
        'scope': 'Synthetic RFM-context surrogate and supervised sidecar. No robot/controller or FP2 performance reproduction.'}
print(json.dumps(results, indent=2))


## 面试追问与实验记录
1. Q 来自哪里，为什么输出只有一个 token？
2. 哪个阶段允许 force loss 更新 compressor？本实验中答案是没有。
3. 修改 d，观察 reconstruction 与下游误差是否同步。
4. 在 validation 上诊断 zero/shuffled context 或 force，解释 OOD 干预的局限。
5. mask/desired wrench 与实际 controller command 有什么区别？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
